# Credit Card Customer Segmentation

This notebook analyzes credit card customer behavior using 18 behavioral variables and compares four clustering algorithms:

- K-means
- Hierarchical Clustering
- Gaussian Mixture Model
- DBSCAN

The goal is to segment customers into 4 groups based on spending, purchase frequency, and recency behavior, evaluate clustering quality, and generate actionable customer personas and marketing strategies.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
DATA_PATH = Path("data/Credit Card Dataset for Clustering.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_CLUSTERS = 4

FEATURES_ALL = [
    "BALANCE",
    "BALANCE_FREQUENCY",
    "PURCHASES",
    "ONEOFF_PURCHASES",
    "INSTALLMENTS_PURCHASES",
    "CASH_ADVANCE",
    "PURCHASES_FREQUENCY",
    "ONEOFF_PURCHASES_FREQUENCY",
    "PURCHASES_INSTALLMENTS_FREQUENCY",
    "CASH_ADVANCE_FREQUENCY",
    "CASH_ADVANCE_TRX",
    "PURCHASES_TRX",
    "CREDIT_LIMIT",
    "PAYMENTS",
    "MINIMUM_PAYMENTS",
    "PRC_FULL_PAYMENT",
    "TENURE",
    "RECENCY_PROXY",
]

## Load and prepare the dataset

The original dataset does not include a direct recency field, so a proxy is created:

**RECENCY_PROXY = 1 - BALANCE_FREQUENCY**

In [ ]:
df = pd.read_csv(DATA_PATH)

df["MINIMUM_PAYMENTS"] = df["MINIMUM_PAYMENTS"].fillna(df["MINIMUM_PAYMENTS"].median())
df["CREDIT_LIMIT"] = df["CREDIT_LIMIT"].fillna(df["CREDIT_LIMIT"].median())
df["RECENCY_PROXY"] = 1 - df["BALANCE_FREQUENCY"].fillna(df["BALANCE_FREQUENCY"].median())

df.head()

In [ ]:
df.describe().T

## Add business-friendly spend and hidden segment labels

In [ ]:
def spending_band(purchases):
    if purchases >= 3000:
        return "High Spending"
    if purchases < 500:
        return "Low Spending"
    return "Mid Spending"

def hidden_segment(row):
    if row["PURCHASES_FREQUENCY"] >= 0.80 and row["PURCHASES"] < 700:
        return "High Engagement, Low Spending"
    if row["BALANCE_FREQUENCY"] < 0.80 and row["PURCHASES"] < 500:
        return "Dormant Low Spenders"
    if row["PURCHASES"] >= 3000:
        return "Premium Spenders"
    return "Core Transactors"

df["spend_band"] = df["PURCHASES"].apply(spending_band)
df["hidden_segment"] = df.apply(hidden_segment, axis=1)

df[["CUST_ID", "PURCHASES", "PURCHASES_FREQUENCY", "BALANCE_FREQUENCY", "spend_band", "hidden_segment"]].head()

## Preprocess numeric features

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, FEATURES_ALL),
    ]
)

X_scaled = preprocessor.fit_transform(df[FEATURES_ALL])
X_scaled.shape

## Fit clustering models

In [ ]:
models = {
    "KMeans": KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE),
    "Hierarchical": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "GaussianMixture": GaussianMixture(n_components=4, covariance_type="full", random_state=RANDOM_STATE),
    "DBSCAN": DBSCAN(eps=0.9, min_samples=12),
}

labels_dict = {}
labels_dict["KMeans"] = models["KMeans"].fit_predict(X_scaled)
labels_dict["Hierarchical"] = models["Hierarchical"].fit_predict(X_scaled)
labels_dict["GaussianMixture"] = models["GaussianMixture"].fit_predict(X_scaled)
labels_dict["DBSCAN"] = models["DBSCAN"].fit_predict(X_scaled)

## Evaluate clustering performance
- Silhouette Score: higher is better
- Davies-Bouldin Index: lower is better

In [ ]:
def evaluate_clustering(X_scaled, labels):
    valid_mask = labels != -1
    X_eval = X_scaled[valid_mask]
    y_eval = labels[valid_mask]

    unique_clusters = np.unique(y_eval)

    if len(unique_clusters) < 2:
        return {
            "silhouette_score": np.nan,
            "davies_bouldin_index": np.nan,
            "n_clusters": int(len(unique_clusters)),
            "noise_points": int(np.sum(labels == -1)),
        }

    return {
        "silhouette_score": float(silhouette_score(X_eval, y_eval)),
        "davies_bouldin_index": float(davies_bouldin_score(X_eval, y_eval)),
        "n_clusters": int(len(unique_clusters)),
        "noise_points": int(np.sum(labels == -1)),
    }

score_rows = []
for algorithm, labels in labels_dict.items():
    metrics = evaluate_clustering(X_scaled, labels)
    score_rows.append({"algorithm": algorithm, **metrics})

scores_df = pd.DataFrame(score_rows)
scores_df.sort_values(["silhouette_score", "davies_bouldin_index"], ascending=[False, True])

## Select the winning algorithm

In [ ]:
valid_scores = scores_df.dropna(subset=["silhouette_score", "davies_bouldin_index"]).copy()
valid_scores["silhouette_rank"] = valid_scores["silhouette_score"].rank(ascending=False, method="min")
valid_scores["db_rank"] = valid_scores["davies_bouldin_index"].rank(ascending=True, method="min")
valid_scores["combined_rank"] = valid_scores["silhouette_rank"] + valid_scores["db_rank"]

winning_algorithm = (
    valid_scores.sort_values(
        ["combined_rank", "silhouette_score", "davies_bouldin_index"],
        ascending=[True, False, True],
    )
    .iloc[0]["algorithm"]
)

winning_algorithm

## Label the dataset with cluster assignments

In [ ]:
for algorithm, labels in labels_dict.items():
    df[f"{algorithm.lower()}_cluster"] = labels

df["winning_cluster"] = labels_dict[winning_algorithm]

df.head()

## Run PCA for 2D visualization support

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(X_scaled)

df["pca_x"] = pca_coords[:, 0]
df["pca_y"] = pca_coords[:, 1]

df[["pca_x", "pca_y"]].head()

## Build cluster personas and marketing strategies

In [ ]:
def assign_persona(row):
    if row["PURCHASES"] >= 3000 and row["PURCHASES_FREQUENCY"] >= 0.75:
        return "Premium Power Users"
    if row["PURCHASES_FREQUENCY"] >= 0.70 and row["PURCHASES"] < 700:
        return "Engaged Low Spenders"
    if row["RECENCY_PROXY"] >= 0.12 and row["PURCHASES"] < 600:
        return "At-Risk Dormant Customers"
    return "Everyday Revolving Spenders"

def marketing_strategy(persona):
    strategies = {
        "Premium Power Users": "Offer premium rewards, loyalty perks, exclusive benefits, and higher credit line upsell campaigns.",
        "Engaged Low Spenders": "Use basket-building offers, category bundles, and spend-threshold cashback to convert engagement into higher spend.",
        "At-Risk Dormant Customers": "Launch win-back campaigns with limited-time credits, reminders, and personalized reactivation incentives.",
        "Everyday Revolving Spenders": "Promote installment plans, autopay nudges, and recurring everyday partner offers.",
    }
    return strategies[persona]

cluster_profiles = (
    df.groupby("winning_cluster")
    .agg(
        customers=("CUST_ID", "count"),
        BALANCE=("BALANCE", "mean"),
        PURCHASES=("PURCHASES", "mean"),
        CREDIT_LIMIT=("CREDIT_LIMIT", "mean"),
        PURCHASES_FREQUENCY=("PURCHASES_FREQUENCY", "mean"),
        RECENCY_PROXY=("RECENCY_PROXY", "mean"),
        TENURE=("TENURE", "mean"),
    )
    .reset_index()
    .sort_values("PURCHASES", ascending=False)
)

cluster_profiles["persona"] = cluster_profiles.apply(assign_persona, axis=1)
cluster_profiles["marketing_strategy"] = cluster_profiles["persona"].apply(marketing_strategy)

cluster_profiles

## Export outputs

In [ ]:
OUTPUT_LABELED = OUTPUT_DIR / "credit_card_customers_labeled.csv"
OUTPUT_MODEL_COMPARISON = OUTPUT_DIR / "clustering_model_comparison.csv"
OUTPUT_CLUSTER_PROFILES = OUTPUT_DIR / "winning_cluster_profiles.csv"
OUTPUT_SUMMARY = OUTPUT_DIR / "credit_card_cluster_summary.json"

scores_df.to_csv(OUTPUT_MODEL_COMPARISON, index=False)
cluster_profiles.to_csv(OUTPUT_CLUSTER_PROFILES, index=False)
df.to_csv(OUTPUT_LABELED, index=False)

summary_payload = {
    "winning_algorithm": winning_algorithm,
    "scores": scores_df.to_dict(orient="records"),
    "cluster_profiles": cluster_profiles.to_dict(orient="records"),
    "notes": [
        "RECENCY_PROXY was derived as 1 - BALANCE_FREQUENCY because the dataset has no direct recency field.",
        "DBSCAN may return noise points labeled -1.",
        "The final labeled dataset uses the best-performing model based on Silhouette Score and Davies-Bouldin Index.",
    ],
}

with open(OUTPUT_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Saved:")
print("-", OUTPUT_MODEL_COMPARISON)
print("-", OUTPUT_CLUSTER_PROFILES)
print("-", OUTPUT_LABELED)
print("-", OUTPUT_SUMMARY)

## Final takeaway

This project demonstrates how unsupervised learning can uncover actionable customer segments for targeted marketing, retention, and upsell strategy. It also shows how model outputs can be translated into business-friendly personas and dashboard-ready insights.